In [1]:
from api_setup import setup
import re
import json

client, model_id = setup()

# Magic string to trigger redacted thinking
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"

In [13]:
# Helper functions


def add_user_message(messages, content):
    if isinstance(content, str):
        user_message = {"role": "user", "content": [{"text": content}]}
    else:
        user_message = {"role": "user", "content": content}
    messages.append(user_message)


def add_assistant_message(messages, content):
    if isinstance(content, str):
        assistant_message = {
            "role": "assistant",
            "content": [{"text": content}],
        }
    else:
        assistant_message = {"role": "assistant", "content": content}

    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    tool_choice="auto",
    text_editor=None,
    thinking=False,
    thinking_budget=1024
):
    params = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {
            "temperature": temperature,
            "stopSequences": stop_sequences,
        },
    }

    if system:
        params["system"] = [{"text": system}]

    tool_choices = {
        "auto": {"auto": {}},
        "any": {"any": {}},
    }
    if tools or text_editor:
        choice = tool_choices.get(tool_choice, {"tool": {"name": tool_choice}})
        params["toolConfig"] = {"tools": tools, "toolChoice": choice}

    additional_inference_fields = {}
    if text_editor:
        additional_inference_fields["tools"] = [
            {
                "type": text_editor,
                "name": "str_replace_editor",
            }
        ]

    if thinking:
        additional_inference_fields["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget
        }

    params["additionalModelRequestFields"] = additional_inference_fields
    response = client.converse(**params)
    parts = response["output"]["message"]["content"]

    return {
        "parts": parts,
        "stop_reason": response["stopReason"],
        "text": "\n".join([p["text"] for p in parts if "text" in p]),
    }

In [15]:
messages = []

add_user_message(
    messages,
    thinking_test_str,

)

result = chat(messages, thinking=True)

result["parts"]

[{'reasoningContent': {'redactedContent': b'EpMICoQBCBMQARgCKkCTx406MUM8V5vPIt22BMcXQ3jGaM9fCPlmU2oDtD0m7I61yXbBglfhI8fYk4HCLmMIGy3MJZRej1Z9C825ap8oMhljbGF1ZGUtaGFpa3UtNC01LTIwMjUxMDAxOABCCHRoaW5raW5nWgw5MzEzMzI4ODg1MTaoAbL+gtcGEgwu8P1baUWouvUHWqcaDIbzDM4CDNMmoic3ECIwlRIKAnwko90SQJjSE3dDa+yM3FTZ5gOZ2xoF+JKfRdhLjP0sPg84K45QeM7WSfB4KrsGNFp+8EEaPo22VTYkGMkGAcZe6U3XN70iFXK8n7xPXpAAU3hYkxvuBhk+Av5+xB0b86pIMgxdH+iMZO8Wd9xcH3QB9t6wh2Bjtk6zHgs+iDryLiKOtqtarUky1t6cNtZiNMxaHQq77tTOAnupekkP8A7Fgu4GaC5tIsQWCoz3sZGU2J08jCHHLD/Hc++bA0ju8xX4jN4AMbPcImcIQVYss2hhgoCSmFgz4auM6c/AnhofjfV0tVaAeRwLX/ffobTk568rzcUBPooSHYqMNBcAAo8P7ReXIqNUhHFsriqcSVHkawWh1lpRZ6V/CNcOmaPymFOKCo7zojDZ9ndi58sKekuTB1AMNXJv+t/faLNTxmPL+xAYc0VZcI+W6OvzXI0m0P/1ryRCduLnuAGBlSW4eEt/S2UHN8MJzJL22rwign/RtpfvXwiGe5G2fLOgawkLIu8p87OzHoYm/riOpCYi1hVPyNaz3M0Kks4kEkRAMEbzuk1nJwPKzMgr0wXJfVnGAjpUj88RHPCQMZyX4pLFPHdBLRtBTvcngk+PcVhvBELxwSh9CtA5UETDWdFQ5t6oa73VDyvs8+ybZ0dbZSXJL4dTEd0wQgMCCLJf/hpIWXu6pdPjbApIWhkMQtPzP0Ci8dAxqxrFNIhU9eCqW7rLD+ux